# ② 검색 — 색인 만들기 · 검색
담당: 검색 · 코드: `rag/retrieval/indexing.py`, `rag/retrieval/retrieval.py`

01 전처리의 `artifacts/chunks.jsonl` 로 Dense(KURE-v1 임베딩 → FAISS)와 BM25(Kiwi 형태소) 색인을 만들어 `artifacts/index/` 에 저장합니다.
검색 전에 OpenAI API로 질문을 재작성합니다(유료). 재작성한 질문으로 Dense·BM25 검색을 하고, 원본 질문에 나온 발주기관 문서 안에서만 찾습니다.
`.env` 에 Langfuse 키가 있으면 검색할 때마다 Langfuse Cloud 에 `retrieve-context` 로 기록됩니다 (없어도 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl
from rag.retrieval.indexing import build_index
from rag.retrieval.retrieval import HybridIndex
from rag.retrieval.rewrite import rewrite_question

cfg = load_config()

## 1. 색인 만들기

In [ ]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
build_index(chunks, cfg, ARTIFACTS / "index")

## 2. 검색해 보기
`config.yaml` 의 `retrieval` (top_k · dense_weight · candidate_k)과 `rewrite` (model · max_tokens · reasoning_effort)를 바꿔 가며 결과를 확인하세요.

In [ ]:
index = HybridIndex(ARTIFACTS, cfg)
QUESTION = "고려대학교 차세대 포털 시스템 사업의 응답 시간 요구사항은?"
rewrite = rewrite_question(QUESTION, cfg["rewrite"])
print(rewrite)
pd.DataFrame(index.search(QUESTION, query=rewrite["question"]))[["score", "title", "page", "kind", "section", "body"]]